# Stage 3: Algorithm Definition & Model Training
# This notebook splits the data chronologically at the year 1990. We define the XGBoost algorithm and apply cost-sensitive weighting to aggressively penalize missed crisis warnings.

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

# 1. Load the processed signals from Notebook 2
df = pd.read_csv('data/processed_signals.csv')
features = ['yield_curve_slope', 'credit_gdp', 'credit_gdp_diff2', 'credit_gdp_cycle', 'yield_curve_cycle', 'cpi', 'unemp', 'debtgdp']

# 2. Chronological Walk-Forward Split (Cutoff: 1990)
cutoff_year = 1990
train_df = df[df['year'] < cutoff_year]
test_df = df[df['year'] >= cutoff_year]

X_train, y_train = train_df[features], train_df['crisisJST']
X_test, y_test = test_df[features], test_df['crisisJST']

# 3. Define the Algorithm
# Calculate scale_pos_weight to handle the imbalanced nature of financial crises
ratio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=ratio, # The cost-sensitive penalty
    random_state=42
)

# 4. Train the Model
print("Training XGBoost Algorithm...")
model.fit(X_train, y_train)

# 5. Evaluate Baseline Accuracy
y_pred_proba = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"Model Trained! PR-AUC Score: {pr_auc:.4f}")

# 6. Save the trained brain
model.save_model("models/xgboost_crisis_model.json")
print("Algorithm saved to disk.")